# Modelado y Optimización de Hiperparámetros
## Predicción de Readmisión Temprana en Pacientes Diabéticos

**Objetivo**: Entrenar y optimizar 4 modelos (Gaussian NB, DT, RF, SVM) con GridSearchCV, aplicar SMOTE para desbalance, y evaluar con métricas apropiadas.

**Contexto**: Dataset ~98k registros, 30-40 features post-codificación, clase objetivo `<30` (~11% desbalanceada).

## 1. Cargar librerías y datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
PALETTE = {'NO': '#2ecc71', '>30': '#f39c12', '<30': '#e74c3c'}

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [2]:
# Cargar dataset limpio del EDA
import os
os.chdir('/workspaces/Proyecto_Final_IA1_UIS')

df_raw = pd.read_csv('diabetic_data.csv', na_values='?')
print(f"✅ Dataset original: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

# Preprocesamiento idéntico al EDA
admission_map = {1:'Emergency',2:'Urgent',3:'Elective',4:'Newborn',5:'Not Available',7:'Trauma Center',8:'Not Mapped'}
discharge_map = {1:'Home',2:'Short-term Hospital',3:'SNF',4:'ICF',5:'Other Inpatient',6:'Home+Health Service',
                 7:'Left AMA',8:'Home IV Provider',9:'Admitted Inpatient',10:'Neonatal Transfer',
                 11:'Expired',12:'Still Patient',13:'Hospice/Home',14:'Hospice/Facility',15:'Medicare Swing Bed',
                 16:'Outpatient Ref (other)',17:'Outpatient Ref (this)',19:'Expired@Home',
                 20:'Expired@Facility',21:'Expired Unknown',22:'Rehab Facility',23:'Long-term Care',
                 24:'Medicaid Nursing',25:'Not Mapped',26:'Unknown'}
admission_src_map = {1:'Physician Referral',2:'Clinic Referral',3:'HMO Referral',4:'Transfer Hospital',
                     5:'Transfer SNF',6:'Transfer Other',7:'Emergency Room',8:'Court/Law',9:'Not Available'}

df = df_raw.copy()
df['admission_type_label'] = df['admission_type_id'].map(admission_map)
df['discharge_disposition_label'] = df['discharge_disposition_id'].map(discharge_map)
df['admission_source_label'] = df['admission_source_id'].map(admission_src_map)

# Filtrado crítico: eliminar fallecidos y en hospicio
IDS_EXCLUIR = [11, 13, 14, 19, 20, 21]
antes = len(df)
df = df[~df['discharge_disposition_id'].isin(IDS_EXCLUIR)].copy()
print(f"✅ Filtrado crítico: {antes - len(df):,} registros eliminados")

# Eliminar columnas con >50% nulos
nulos_pct = (df.isnull().sum() / len(df)) * 100
cols_eliminar = nulos_pct[nulos_pct > 50].index.tolist()
df.drop(columns=cols_eliminar, inplace=True)
print(f"✅ Columnas eliminadas por >50% nulos: {cols_eliminar}")

print(f"📊 Dataset limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")

✅ Dataset original: 101,766 filas × 50 columnas
✅ Filtrado crítico: 2,423 registros eliminados
✅ Columnas eliminadas por >50% nulos: ['weight', 'max_glu_serum', 'A1Cresult']
📊 Dataset limpio: 99,343 filas × 50 columnas


## 2. Preparación de Features - Codificación y Selección

In [3]:
# Mapeos para variables ordinales y binarias
age_map = {
    '[0-10)': 0, '[10-20)': 1, '[20-30)': 2, '[30-40)': 3, '[40-50)': 4,
    '[50-60)': 5, '[60-70)': 6, '[70-80)': 7, '[80-90)': 8, '[90-100)': 9
}

insulin_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
binary_map = {'No': 0, 'Yes': 1}

# Aplicar mapeos
df['age_encoded'] = df['age'].map(age_map)
df['insulin_encoded'] = df['insulin'].map(insulin_map)
df['change_encoded'] = df['change'].map(binary_map)
df['diabetesMed_encoded'] = df['diabetesMed'].map(binary_map)

print("✅ Variables ordinales y binarias codificadas")
print(f"Age: {df['age_encoded'].unique()}")
print(f"Insulin: {df['insulin_encoded'].unique()}")
print(f"Change: {df['change_encoded'].unique()}")
print(f"DiabetesMed: {df['diabetesMed_encoded'].unique()}")

✅ Variables ordinales y binarias codificadas
Age: [0 1 2 3 4 5 6 7 8 9]
Insulin: [0 2 1 3]
Change: [ 0. nan]
DiabetesMed: [0 1]


In [4]:
# Seleccionar features para modelado
# Numéricas (sin scaling aún)
numeric_features = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_diagnoses',
    'number_outpatient', 'number_emergency', 'number_inpatient'
]

# Ordinales y binarias (ya codificadas)
ordinal_binary_features = ['age_encoded', 'insulin_encoded', 'change_encoded', 'diabetesMed_encoded']

# Nominales para OneHotEncoding
categorical_features = ['gender', 'race', 'admission_type_label', 'discharge_disposition_label']

# Variable objetivo
target = 'readmitted'

# Validar que todas las features existan
all_features = numeric_features + ordinal_binary_features + categorical_features
missing_features = [f for f in all_features if f not in df.columns]
if missing_features:
    print(f"⚠️ Features faltantes: {missing_features}")
else:
    print(f"✅ Todas las features disponibles")

# Imputación de nulos restantes
for col in numeric_features + ordinal_binary_features:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        
for col in categorical_features:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"✅ Nulos imputados")
print(f"📊 Features numéricas: {len(numeric_features)}")
print(f"📊 Features ordinales/binarias: {len(ordinal_binary_features)}")
print(f"📊 Features categóricas nominales: {len(categorical_features)}")
print(f"📊 Total features: {len(all_features)}")

✅ Todas las features disponibles
✅ Nulos imputados
📊 Features numéricas: 8
📊 Features ordinales/binarias: 4
📊 Features categóricas nominales: 4
📊 Total features: 16


## 3. Codificación OneHotEncoding para Nominales + StandardScaler

In [ ]:
# Construcción de ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric_scaler', StandardScaler(), numeric_features),
        ('ordinal_binary_identity', 'passthrough', ordinal_binary_features),
        ('nominal_onehot', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

# Preparar X e y
X = df[all_features].copy()
y = df[target].copy()

# Aplicar transformaciones
X_transformed = preprocessor.fit_transform(X)

# Convertir a DataFrame y manejar cualquier NaN
X_transformed = np.nan_to_num(X_transformed, nan=0.0)

# Obtener nombres de features post-transformación (auxiliar para interpretabilidad)
feature_names = (numeric_features + ordinal_binary_features + 
                 list(preprocessor.named_transformers_['nominal_onehot'].get_feature_names_out(categorical_features)))

print(f"✅ Transformaciones aplicadas")
print(f"   - Numéricas: StandardScaler aplicado (media=0, desv=1)")
print(f"   - Ordinales/Binarias: Sin cambios (ya codificadas 0-3)")
print(f"   - Nominales: OneHotEncoding (drop='first' para evitar multicolinealidad)")
print(f"\n📊 Matriz X: {X_transformed.shape[0]:,} × {X_transformed.shape[1]} features")
print(f"📊 Target y: {y.shape[0]:,} muestras")
print(f"📊 NaN en X_transformed: {np.isnan(X_transformed).sum()}")
print(f"\nDistribución de clases:")
print(y.value_counts())
print(f"\nProporción:")
print((y.value_counts() / len(y) * 100).round(2))

✅ Transformaciones aplicadas
   - Numéricas: StandardScaler aplicado (media=0, desv=1)
   - Ordinales/Binarias: Sin cambios (ya codificadas 0-3)
   - Nominales: OneHotEncoding (drop='first' para evitar multicolinealidad)

📊 Matriz X: 99,343 × 44 features
📊 Target y: 99,343 muestras

Distribución de clases:
readmitted
NO     52527
>30    35502
<30    11314
Name: count, dtype: int64

Proporción:
readmitted
NO     52.87
>30    35.74
<30    11.39
Name: count, dtype: float64


## 4. Split Train/Test Estratificado

In [6]:
# Split 75/25 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print(f"✅ Split 75/25 estratificado aplicado")
print(f"\n📊 ENTRENAMIENTO ({len(X_train):,} muestras):")
print(y_train.value_counts())
print(f"Proporción: {(y_train.value_counts() / len(y_train) * 100).round(2).to_dict()}")

print(f"\n📊 PRUEBA ({len(X_test):,} muestras):")
print(y_test.value_counts())
print(f"Proporción: {(y_test.value_counts() / len(y_test) * 100).round(2).to_dict()}")

print(f"\n✅ Proporciones idénticas → Stratificación exitosa")

✅ Split 75/25 estratificado aplicado

📊 ENTRENAMIENTO (74,507 muestras):
readmitted
NO     39395
>30    26626
<30     8486
Name: count, dtype: int64
Proporción: {'NO': 52.87, '>30': 35.74, '<30': 11.39}

📊 PRUEBA (24,836 muestras):
readmitted
NO     13132
>30     8876
<30     2828
Name: count, dtype: int64
Proporción: {'NO': 52.87, '>30': 35.74, '<30': 11.39}

✅ Proporciones idénticas → Stratificación exitosa


## 5. Aplicar SMOTE en Conjunto de Entrenamiento

In [7]:
# Aplicar SMOTE: llevar clase <30 a 30% del dataset
smote = SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"✅ SMOTE aplicado (sampling_strategy=0.3)")
print(f"\n📊 ANTES de SMOTE ({len(X_train):,} muestras):")
print(y_train.value_counts())
print(f"Proporción: {(y_train.value_counts() / len(y_train) * 100).round(2).to_dict()}")

print(f"\n📊 DESPUÉS de SMOTE ({len(X_train_balanced):,} muestras):")
print(y_train_balanced.value_counts())
print(f"Proporción: {(y_train_balanced.value_counts() / len(y_train_balanced) * 100).round(2).to_dict()}")

print(f"\nNuevas muestras generadas: {len(X_train_balanced) - len(X_train):,}")
print(f"Aumento: {((len(X_train_balanced) - len(X_train)) / len(X_train) * 100):.1f}%")

ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 6. Definición de Métrica Personalizada y Config de GridSearch

In [ ]:
from sklearn.metrics import make_scorer

# Scorer personalizado: F1-macro (principal)
# Se optimiza F1-macro durante GridSearch, se reportan todas las métricas al final
scoring = {
    'f1_macro': 'f1_macro',
    'recall_macro': 'recall_macro',
    'precision_macro': 'precision_macro',
    'accuracy': 'accuracy'
}

# CV estratificado
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("✅ Scorer y CV strategy configurados")
print(f"   - Scorer primario: F1-macro (balancea todas las clases)")
print(f"   - Scorer secundarios: Recall_macro, Precision_macro, Accuracy")
print(f"   - CV: 5-Fold estratificado (mantiene proporción de clases en cada fold)")
print(f"   - n_jobs=-1 (usar todos los cores disponibles)")

## 7. Modelo 1: Gaussian Naive Bayes

In [ ]:
import time

print("="*80)
print("MODELO 1: GAUSSIAN NAIVE BAYES")
print("="*80)

param_grid_nb = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

gnb = GaussianNB()
grid_nb = GridSearchCV(
    gnb,
    param_grid_nb,
    cv=cv_strategy,
    scoring=scoring,
    refit='f1_macro',  # Refit con mejor F1-macro
    n_jobs=-1,
    verbose=1
)

print(f"\n🔍 Buscando mejores hiperparámetros...")
start = time.time()
grid_nb.fit(X_train_balanced, y_train_balanced)
elapsed = time.time() - start

print(f"\n✅ GridSearch completado en {elapsed:.2f}s")
print(f"\n📊 Mejor F1-macro: {grid_nb.best_score_:.4f}")
print(f"📊 Mejores parámetros: {grid_nb.best_params_}")
print(f"\n📌 Top 5 configuraciones:")
results_nb = pd.DataFrame(grid_nb.cv_results_).sort_values('rank_test_f1_macro').head(5)
for col in ['param_var_smoothing', 'mean_test_f1_macro', 'mean_test_recall_macro', 'mean_test_accuracy']:
    if col in results_nb.columns:
        print(f"   {col}: {results_nb[col].values}")

# Predicción en test
y_pred_nb = grid_nb.predict(X_test)
y_pred_proba_nb = grid_nb.predict_proba(X_test)

# Métricas en test
acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
precision_nb = precision_score(y_test, y_pred_nb, average='macro')

print(f"\n📈 RESULTADOS EN TEST:")
print(f"   Accuracy:        {acc_nb:.4f}")
print(f"   F1-macro:        {f1_nb:.4f}")
print(f"   Recall-macro:    {recall_nb:.4f}")
print(f"   Precision-macro: {precision_nb:.4f}")
print(f"   Tiempo entrenamiento: {elapsed:.2f}s")

## 8. Modelo 2: Decision Tree

In [ ]:
print("="*80)
print("MODELO 2: DECISION TREE")
print("="*80)

param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 4, 8, 16]
}

dt = DecisionTreeClassifier(random_state=42)
grid_dt = GridSearchCV(
    dt,
    param_grid_dt,
    cv=cv_strategy,
    scoring=scoring,
    refit='f1_macro',
    n_jobs=-1,
    verbose=1
)

print(f"\n🔍 Buscando mejores hiperparámetros...")
print(f"   Total combinaciones a evaluar: {len(param_grid_dt['criterion']) * len(param_grid_dt['max_depth']) * len(param_grid_dt['min_samples_split']) * len(param_grid_dt['min_samples_leaf'])}")
start = time.time()
grid_dt.fit(X_train_balanced, y_train_balanced)
elapsed = time.time() - start

print(f"\n✅ GridSearch completado en {elapsed:.2f}s")
print(f"\n📊 Mejor F1-macro: {grid_dt.best_score_:.4f}")
print(f"📊 Mejores parámetros: {grid_dt.best_params_}")

# Predicción en test
y_pred_dt = grid_dt.predict(X_test)
y_pred_proba_dt = grid_dt.predict_proba(X_test)

# Métricas en test
acc_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt, average='macro')
recall_dt = recall_score(y_test, y_pred_dt, average='macro')
precision_dt = precision_score(y_test, y_pred_dt, average='macro')

print(f"\n📈 RESULTADOS EN TEST:")
print(f"   Accuracy:        {acc_dt:.4f}")
print(f"   F1-macro:        {f1_dt:.4f}")
print(f"   Recall-macro:    {recall_dt:.4f}")
print(f"   Precision-macro: {precision_dt:.4f}")
print(f"   Tiempo entrenamiento: {elapsed:.2f}s")

## 9. Modelo 3: Random Forest

In [ ]:
print("="*80)
print("MODELO 3: RANDOM FOREST")
print("="*80)

# GridSearch más exhaustivo pero se puede cambiar a RandomizedSearchCV si es muy lento
param_grid_rf = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 4, 8],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=cv_strategy,
    scoring=scoring,
    refit='f1_macro',
    n_jobs=-1,
    verbose=1
)

print(f"\n🔍 Buscando mejores hiperparámetros...")
total_combos = (
    len(param_grid_rf['n_estimators']) *
    len(param_grid_rf['max_depth']) *
    len(param_grid_rf['min_samples_split']) *
    len(param_grid_rf['min_samples_leaf']) *
    len(param_grid_rf['max_features']) *
    len(param_grid_rf['class_weight'])
)
print(f"   Total combinaciones a evaluar: {total_combos}")

start = time.time()
grid_rf.fit(X_train_balanced, y_train_balanced)
elapsed = time.time() - start

print(f"\n✅ GridSearch completado en {elapsed:.2f}s")
print(f"\n📊 Mejor F1-macro: {grid_rf.best_score_:.4f}")
print(f"📊 Mejores parámetros: {grid_rf.best_params_}")

# Predicción en test
y_pred_rf = grid_rf.predict(X_test)
y_pred_proba_rf = grid_rf.predict_proba(X_test)

# Métricas en test
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='macro')
recall_rf = recall_score(y_test, y_pred_rf, average='macro')
precision_rf = precision_score(y_test, y_pred_rf, average='macro')

print(f"\n📈 RESULTADOS EN TEST:")
print(f"   Accuracy:        {acc_rf:.4f}")
print(f"   F1-macro:        {f1_rf:.4f}")
print(f"   Recall-macro:    {recall_rf:.4f}")
print(f"   Precision-macro: {precision_rf:.4f}")
print(f"   Tiempo entrenamiento: {elapsed:.2f}s")

## 10. Modelo 4: Support Vector Machine (SVM) - RandomizedSearchCV

In [ ]:
print("="*80)
print("MODELO 4: SUPPORT VECTOR MACHINE (SVM)")
print("="*80)

# Para SVM usamos RandomizedSearchCV por ser más rápido
param_dist_svm = {
    'C': [0.1, 1, 10, 100, 1000],
    'kernel': ['rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'degree': [2, 3],
    'class_weight': ['balanced', None]
}

svm = SVC(random_state=42)
grid_svm = RandomizedSearchCV(
    svm,
    param_dist_svm,
    n_iter=50,  # Evaluar 50 combinaciones aleatorias
    cv=cv_strategy,
    scoring=scoring,
    refit='f1_macro',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

print(f"\n🔍 Buscando mejores hiperparámetros (RandomizedSearchCV con n_iter=50)...")
start = time.time()
grid_svm.fit(X_train_balanced, y_train_balanced)
elapsed = time.time() - start

print(f"\n✅ RandomizedSearchCV completado en {elapsed:.2f}s")
print(f"\n📊 Mejor F1-macro: {grid_svm.best_score_:.4f}")
print(f"📊 Mejores parámetros: {grid_svm.best_params_}")

# Predicción en test
y_pred_svm = grid_svm.predict(X_test)
y_pred_proba_svm = grid_svm.decision_function(X_test)  # SVM usa decision_function

# Métricas en test
acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='macro')
recall_svm = recall_score(y_test, y_pred_svm, average='macro')
precision_svm = precision_score(y_test, y_pred_svm, average='macro')

print(f"\n📈 RESULTADOS EN TEST:")
print(f"   Accuracy:        {acc_svm:.4f}")
print(f"   F1-macro:        {f1_svm:.4f}")
print(f"   Recall-macro:    {recall_svm:.4f}")
print(f"   Precision-macro: {precision_svm:.4f}")
print(f"   Tiempo entrenamiento: {elapsed:.2f}s")

## 11. Comparativa de Modelos

In [ ]:
# Tabla comparativa
comparison_df = pd.DataFrame({
    'Modelo': ['Gaussian NB', 'Decision Tree', 'Random Forest', 'SVM'],
    'Accuracy': [acc_nb, acc_dt, acc_rf, acc_svm],
    'F1-macro': [f1_nb, f1_dt, f1_rf, f1_svm],
    'Recall-macro': [recall_nb, recall_dt, recall_rf, recall_svm],
    'Precision-macro': [precision_nb, precision_dt, precision_rf, precision_svm]
})

print("\n" + "="*80)
print("COMPARATIVA DE MODELOS - CONJUNTO DE PRUEBA")
print("="*80)
print(comparison_df.to_string(index=False))
print("\n✅ Mejor modelo por F1-macro:", comparison_df.loc[comparison_df['F1-macro'].idxmax(), 'Modelo'])

# Visualización
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = ['Accuracy', 'F1-macro', 'Recall-macro', 'Precision-macro']

for idx, metric in enumerate(metrics):
    axes[idx].bar(comparison_df['Modelo'], comparison_df[metric], color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
    axes[idx].set_title(metric, fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Score')
    axes[idx].tick_params(axis='x', rotation=30)
    axes[idx].set_ylim([0, 1])
    for i, v in enumerate(comparison_df[metric]):
        axes[idx].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Comparativa de Modelos en Conjunto de Prueba', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 12. Análisis Profundo - Matrices de Confusión y Reportes

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Matrices de confusión para los 4 modelos
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

models = [
    ('Gaussian NB', y_pred_nb, grid_nb, 0),
    ('Decision Tree', y_pred_dt, grid_dt, 1),
    ('Random Forest', y_pred_rf, grid_rf, 2),
    ('SVM', y_pred_svm, grid_svm, 3)
]

for name, y_pred, model, idx in models:
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['NO', '>30', '<30']).plot(ax=axes[idx], cmap='Blues')
    axes[idx].set_title(f'{name} - F1-macro: {f1_score(y_test, y_pred, average="macro"):.4f}', fontweight='bold')

plt.suptitle('Matrices de Confusión - Conjunto de Prueba', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("\n📊 Análisis de Matrices de Confusión:")
print("   - Filas: Clase verdadera")
print("   - Columnas: Clase predicha")
print("   - Diagonal: Predicciones correctas")
print("   - Fuera de diagonal: Errores")

## 13. Reportes Detallados por Clase

In [ ]:
print("\n" + "="*80)
print("REPORTES DETALLADOS POR CLASE")
print("="*80)

for name, y_pred, *_ in models:
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred, target_names=['NO', '>30', '<30'], digits=4))

## 14. Feature Importance (Random Forest)

In [ ]:
# Feature importance del mejor Random Forest
best_rf = grid_rf.best_estimator_
importances = best_rf.feature_importances_

# Top 20 features
top_indices = np.argsort(importances)[-20:][::-1]
top_importances = importances[top_indices]

print("\n📊 TOP 20 FEATURES IMPORTANTES (Random Forest):")
for i, idx in enumerate(top_indices[:20], 1):
    print(f"   {i:2d}. Feature {idx}: {top_importances[i-1]:.4f}")

# Visualización
plt.figure(figsize=(12, 8))
plt.barh([f'Feature {idx}' for idx in top_indices[:20]], top_importances[:20])
plt.xlabel('Importancia')
plt.title('Top 20 Features - Random Forest', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()